# L2c Example: Hexadecimal Numbers and Text Representation

This example connects dictionaries, sets, strings, and loops to the numerical representation of ASCII and Unicode characters.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
> * __Move between characters and their numbers:__ Map ASCII and Unicode characters to their integer code points and back, and explain why an encoding is a separate question from the code point itself.
> * __Use collections as lookup machinery:__ Represent a character collection with a set and a digit table with a dictionary, choosing each for the operation it makes cheap.
> * __Implement a base conversion:__ Convert a nonnegative base-10 integer to hexadecimal digits with an explicit algorithm, and format the result in the standard `U+XXXX` notation.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

This example uses [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) and [the `PrettyTables.jl` package](https://github.com/ronisbr/PrettyTables.jl) to display the character tables. The course environment also loads [the `Unicode` standard library](https://docs.julialang.org/en/v1/stdlib/Unicode/), which this notebook discusses but does not call. The course environment also loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl); this notebook does not need it.

___

##  Task 1: Explore the ASCII Character Set
Let's start by considering the representation of the characters in the 7-bit ASCII character system. 

> __Fun fact__: While 7-bit ASCII has been replaced a long time ago, it still lives on in the modern [Unicode system](https://en.wikipedia.org/wiki/Unicode). The original ASCII characters are encoded as the first 128 characters (`0`$\rightarrow$`127`) in [the Unicode system](https://en.wikipedia.org/wiki/Unicode).

Let's start by building the `ascii_char_dictionary::Dict{Int64, Char}` dictionary, which is a mapping between the ASCII character index (an integer) and the character value, which [is type `c::Char` in Julia](https://docs.julialang.org/en/v1/base/strings/#Core.Char).

> __Caution:__ This logic involves some advanced tools and techniques we have yet to discuss. You can skip the implementation details for now; we'll come back to them later. However, there is one interesting method, namely [the `convert(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.convert). This is a direct way to convert between the characters that you and I see and the integers that the computer understands.

What do we see?

In [ ]:
ascii_char_dictionary = let

    # initialize -
    ascii_char_dictionary = Dict{Int64, Char}(); # storage for index - character map
    ASCII_character_range = range(0,stop=127,step=1) |> collect; # 7-bit ASCII indexes

    # main loop -
    for i ∈ eachindex(ASCII_character_range)
        my_ascii_char_index = ASCII_character_range[i];
        c = convert(Char, my_ascii_char_index) # hmmm. This is an interesting function.
        ascii_char_dictionary[my_ascii_char_index] = c;
    end
    ascii_char_dictionary;
end

`Unhide` the code block to see how we build a table of the ASCII characters using [the `pretty_table(...)` function exported by the PrettyTables.jl package](https://github.com/ronisbr/PrettyTables.jl). Let's look at what the `ASCII` characters are. 
> __Caution:__ This logic involves some advanced tools and techniques we have yet to discuss. You can skip the implementation details for now; we'll come back to them later. TLDR: We build the rows in the table using a [`for-loop`](https://docs.julialang.org/en/v1/base/base/#for) and [the `convert(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.convert), where we push the data for a row into a [DataFrame](https://dataframes.juliadata.org/stable/), and then display the data by [calling the `pretty_table(...)` function](https://github.com/ronisbr/PrettyTables.jl).

So what do we see?

In [ ]:
let
    ASCII_index_array = keys(ascii_char_dictionary) |> collect |> sort;
    character_table_df = DataFrame();
    for i ∈ eachindex(ASCII_index_array)
        my_ascii_char_index = ASCII_index_array[i];
        c = ascii_char_dictionary[my_ascii_char_index];

        row = (
            i = my_ascii_char_index,
            character = c
        ); # -> what is going on here? This is a cool type called a NamedTuple ...
        push!(character_table_df,row); # push! -> what is going on here?
    end
    pretty_table(character_table_df)
end

In the code block above, we explicitly called [the `convert(...)` function](https://docs.julialang.org/en/v1/manual/conversion-and-promotion/) to convert an [`Int`](https://docs.julialang.org/en/v1/manual/integers-and-floating-point-numbers/#Integers) to a [`Char` type](https://docs.julialang.org/en/v1/base/strings/#Core.Char). That call spelled out what the constructor syntax already does for you: `Int(c)` and `Char(i)` convert between the two directly, and Julia applies conversions like these on its own wherever the target type is unambiguous. For example:

In [ ]:
'+' |> Int # wow. This seems a little magical. What is going on here?? (notice the single quotes), pick a different char?

In [ ]:
Int('+') # Unpack the |> We can convert between ASCII characters and Int! 

In [ ]:
Char(43)

### Limitations of ASCII
Wow! ASCII seems pretty cool. It only uses 7 bits (which is super simple) to encode 128 characters (numbers, small and capital letters, some math symbols, etc). Why would we ever need anything beyond these 128 characters? As it turns out, there are many reasons. 

Let's look at three:
* The original 7-bit ASCII was limited to 128 characters (0-127), so it couldn't represent characters beyond basic English letters, digits, punctuation, and control codes.
* The original 7-bit ASCII character set offered no native support for accented or non-Latin characters (e.g., Cyrillic, Greek, and Arabic), hindering internationalization. It only represented English.
* The original 7-bit ASCII character set omitted modern typographic symbols, mathematical glyphs, and emojis, making it inadequate for rich text or graphical communication. For example, it can't be used to represent rich mathematical text. Bummer!

These limitations, and others, led to the development of [the Unicode system](https://en.wikipedia.org/wiki/Unicode).

___

## Task 2: Explore the Unicode Character Set
Unlike the original 7-bit ASCII character set, which has only 128 characters, the modern [Unicode system](https://en.wikipedia.org/wiki/Unicode) _can encode_ 1,114,112 possible characters (code points). However, as of [Unicode Standard version 16.0 (released September 10, 2024)](https://www.unicode.org/versions/Unicode16.0.0/), only 154,998 of those possible code points are assigned to unique characters; various characters in many languages, a much larger family of mathematical symbols, and [emoji](https://en.wikipedia.org/wiki/Emoji) of all sorts are currently assigned. There is plenty of room to grow!

Technical:
* The [Unicode system](https://en.wikipedia.org/wiki/Unicode) assigns every character an integer code point, and an encoding such as UTF-8 decides how that number turns into bytes, using up to 4 bytes per character. Code points are conventionally __written__ in base $b = 16$ (hexadecimal), but that is notation for our benefit rather than the numbering the standard itself is indexed by. Hexadecimal numbers are used as a convenience; they are much shorter than their binary equivalents and, thus, are easier for us to read and write than long sequences of 0s and 1s. While Unicode code points are ultimately stored as binary values, hexadecimal provides a direct and convenient way for us (humans) to represent these values.
* Julia has [built-in Unicode support in the standard library](https://docs.julialang.org/en/v1/stdlib/Unicode/#Unicode); thus, we can work with these characters in our programs. For more information on the specific Unicode primitives supported by Julia, check out the [Julia documentation](https://docs.julialang.org/en/v1/manual/unicode-input/).

Let's do some math with emojis (for fun):

In [ ]:
🌽 = 16; # corn = 16 \:corn: then tab
🍣 = 4; # sushi = 4 \:sushi: then tab

We can perform operations with these emoji variables:

In [ ]:
🌽 + 🍣  # addition?

In [ ]:
(🌽 * 🍣) # multiplication?

We can also perform logical comparisons. Because Julia has built-in Unicode support, we can use Unicode mathematical symbols to write functional code that resembles standard mathematical notation.
For example, consider `🌽 = 16` and `🍣 = 4`. We know that $🌽\geq{🍣}$ should return `true`.

Let's write the `greater than or equal to` comparison using those Unicode names:

In [ ]:
🌽 ≥ 🍣 # logical comparison? (\geq then tab)

We can also determine whether an item is in a given collection. For example, take a character set $\mathbb{C}$ and ask whether $c\in\mathbb{C}$, where $\in$ denotes the `element of` operation and `c` is some test character.

> Use the [Set data structure](https://docs.julialang.org/en/v1/base/collections/#Base.Set) to do this example. [`Set` is a collection type](https://docs.julialang.org/en/v1/base/collections/#Base.Set) (included in most modern languages) that holds items; sets are `unique` but do not maintain order.

The `C::Set{Char}` variable below holds six letters, one control character, and one emoji, which shows that a `Set{Char}` accepts any `Char` rather than only printable ASCII. A set stores each item once and keeps no order, so the printed result may not match the insertion sequence:

In [ ]:
C = let
    C = Set{Char}(); # empty set with Char types
    push!(C,'A'); # add a `A` to set C 
    push!(C,'B'); # ... `B` ...
    push!(C,'Q'); # ... `Q` ...
    push!(C,'R'); # ... `R` ...
    push!(C,'S'); # ... `S` ...
    push!(C,'T'); # ... `T` ...
    push!(C,2 |> Char); # a non-printing control character is still a Char
    push!(C,'🌽') # ... and so is an emoji

    C # return
end

Specify the test character `c`:

In [ ]:
c = '🌽'; # Do we have corn in the set ℂ?

Check if $c\in\mathbb{C}$:

In [ ]:
c ∈ C # ∈ => \in then tab

The Unicode character set is extensive and powerful, and Julia's support for Unicode is quite advanced. But how are these characters related to base-16 numbers? Let's explore this connection next.

___

## Task 3: A deeper dive into Unicode Strings and Codepoints
The built-in [Julia `String` type](https://docs.julialang.org/en/v1/base/strings/) is similar (in some ways) to the traditional text model in languages like [C](https://en.wikipedia.org/wiki/C_(programming_language)), namely, a [`String`](https://docs.julialang.org/en/v1/base/strings/) is an ordered, immutable sequence of characters. It is not an array of `Char`, though. Julia stores a `String` as UTF-8 bytes, so one character can occupy several bytes and string indices count bytes rather than characters: ask for `"a🍣b"[3]` and you get [a `StringIndexError`](https://docs.julialang.org/en/v1/base/base/#Base.StringIndexError), because position 3 lands inside the emoji. That is why we use [the `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D) on a string when we want an actual `Vector{Char}`.

Let's play around with a `test_string_ascii::String`:

In [ ]:
test_string_ascii = "Test String in Julia (notice the double quotes). Python uses both single and double quotes for Strings. 😒";

We convert `test_string_ascii::String` to an `Array{Char,1}` collection using [the `collect(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.collect-Tuple%7BAny%7D). Each character has a `U+xxxx` type code associated with it. That is a base 16 Unicode code point!
> __What?!?__ In a `U+xxxx` unicode code point, `U+` is followed by a hexadecimal number `xx..x` of at least four digits, uniquely identifying a character in the Unicode standard. Characters above `U+FFFF`, such as the sushi emoji at `U+1F363`, use the five or six digits they need. However, (really) the `U+xxxx` points are just integers! Thus, we should be able to interconvert between the `U+xxxx` and integer representations.

Let's check that out!

In [ ]:
character_array_test_string_ascii = test_string_ascii |> collect

### How do we calculate a codepoint?
First, let's compute the code point for an example Unicode character. We'll use the Greek lunate epsilon `ϵ`, and store it in the `test_unicode_char::Char` variable. (Swap in any character you like; the algorithm below does not care which one.)

In [ ]:
test_unicode_char = 'ϵ' # want to select another Unicode character?

To get the base 10 index of a character, convert it to an integer. `Int(c)` does exactly that, and [the `codepoint(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.codepoint) is the self-documenting spelling of the same idea.

> __Not `julia_chartransform`:__ [The `Unicode.julia_chartransform(...)` function](https://docs.julialang.org/en/v1/stdlib/Unicode/#Unicode.julia_chartransform) is sometimes mistaken for a code-point accessor. It is not. It applies the normalization Julia uses when parsing identifiers, so it returns a `Char`, and for some inputs a __different__ `Char`: it maps `µ` (`U+00B5`) to `μ` (`U+03BC`). Converting that result to an `Int` would report `956` for a character whose code point is `181`. It happens to be the identity for `ϵ`, which is exactly what makes the mistake hard to catch.

So what is the base 10 value for `test_unicode_char::Char`?

In [ ]:
test_char_index = test_unicode_char |> Int # the code point as a base 10 integer

[The `codepoint(...)` function](https://docs.julialang.org/en/v1/base/strings/#Base.codepoint) returns the same number as an unsigned integer, and says out loud what it is asking for:

In [ ]:
codepoint(test_unicode_char) # same value, self-documenting name

So we can get the base 10 representation of the Unicode character. Now let's go from base 10 to the `U+xxxx` representation. Convert the base 10 value to a base 16 (hexadecimal) number and then convert that to the [Unicode index](https://en.wikipedia.org/wiki/Unicode) format. 
> __Hexadecimal digits__: Hexadecimal numbers use decimal digits $(0,1,\dots,9)$ and six extra symbols; the letters `A`, `B`, `C`, `D`, `E`, and `F`, where hexadecimal `A` = decimal 10, through hexadecimal `F` = decimal 15.

Let's start by building the `hexadecimal_digits_dictionary::Dict{Int64, Char}` dictionary which maps the index of a digit to the digit, e.g., `13 => D`.

In [ ]:
hexadecimal_digits_dictionary = let

    # initialize 0
    hexadecimal_digits_dictionary = Dict{Int,Char}()
    base = 16; # what base are using?

    # loop: process each digit 0 -> 15
    for i ∈ 0:(base - 1)
        hexadecimal_digits_dictionary[i] = '0' + i |> Char # hmmm! This is whacky; why does this work?
        if (i > 9)
            hexadecimal_digits_dictionary[i] = 'A' + (i - 10) |> Char 
        end
    end
    hexadecimal_digits_dictionary # return
end

#### Algorithm
Next, let's specify and implement our first algorithm to convert a base 10 number into a Unicode character code. [Unicode](https://en.wikipedia.org/wiki/Unicode) writes a code point as `U+` followed by the hexadecimal value, left-padded with zeros to a minimum of four digits. Code points above `U+FFFF`, such as the emoji, simply use the five or six digits they need.

The `my_code_point::String` variable below holds the finished `U+XXXX` string produced by this algorithm.

__Initialize__: Take a nonnegative integer $x\in\mathbb{Z}_{\geq{0}}$ and the `hexadecimal_digits_dictionary::Dict{Int64, Char}`. Set $q\gets{x}$ and let $R$ be an empty remainder array.

While $q\neq{0}$ __do__:
1.  Record the remainder $r = q \bmod 16$ by appending it to $R$.
2.  Replace $q$ with the quotient $\lfloor q/16 \rfloor$.

The remainders come out least significant digit first, so read $R$ __backwards__, looking each value up in the `hexadecimal_digits_dictionary` to get its hexadecimal digit. Left-pad the result with `0` characters until it is at least four digits long, then prepend `U+`.

We've implemented this algorithm below, does it work (do we get back the proper `U+xx..x` code?)

In [ ]:
my_code_point = let

    q = test_char_index; 
    remainder_array = Array{Int64,1}();
    while (q != 0)
        r = rem(q,16)
        q = div(q,16)
        push!(remainder_array,r)
    end

    my_code_point = "";
    for i ∈ reverse(remainder_array)
        tmp = hexadecimal_digits_dictionary[i];
        my_code_point *= tmp |> Char;
    end

    # left pad with zeros to get a 4-digit code
    my_code_point = lpad(my_code_point, 4, '0') |> x-> "U+"*x
end

___

## Tests
In the code block below, we check some values in your notebook and give you feedback on which items are correct or different. `Unhide` the code block below (if you are curious) about how we implemented the tests and what we are testing.

In [ ]:
@testset verbose = true "CHEME 4/5800 L2c Example Test Suite" begin

    @testset "ASCII Character Dictionary" begin
        # Test that ASCII dictionary is created correctly
        ascii_dict = let
            ascii_dict = Dict{Int64, Char}()
            ASCII_character_range = range(0,stop=127,step=1) |> collect
            for i ∈ eachindex(ASCII_character_range)
                my_ascii_char_index = ASCII_character_range[i]
                c = convert(Char, my_ascii_char_index)
                ascii_dict[my_ascii_char_index] = c
            end
            ascii_dict
        end
        
        @test length(ascii_dict) == 128
        @test ascii_dict[65] == 'A'
        @test ascii_dict[97] == 'a'
        @test ascii_dict[48] == '0'
        @test haskey(ascii_dict, 127)
    end

    @testset "Character to Integer Conversion" begin
        @test Int('A') == 65
        @test Int('a') == 97
        @test Int('0') == 48
        @test Int('🍣') == 127843  # Sushi emoji code point
    end

    @testset "Hexadecimal Digits Dictionary" begin
        hex_dict = let
            hex_dict = Dict{Int,Char}()
            base = 16
            for i ∈ 0:(base - 1)
                hex_dict[i] = '0' + i |> Char
                if (i > 9)
                    hex_dict[i] = 'A' + (i - 10) |> Char 
                end
            end
            hex_dict
        end
        
        @test length(hex_dict) == 16
        @test hex_dict[0] == '0'
        @test hex_dict[9] == '9'
        @test hex_dict[10] == 'A'
        @test hex_dict[15] == 'F'
    end

    @testset "Unicode Code Point Conversion" begin
        # Test the algorithm for converting base 10 to hexadecimal Unicode code point
        test_char = '🍣'
        test_char_index = Int(test_char)
        
        # Expected code point for sushi emoji
        @test test_char_index == 127843
        
        # Test the conversion algorithm
        hex_dict = let
            hex_dict = Dict{Int,Char}()
            base = 16
            for i ∈ 0:(base - 1)
                hex_dict[i] = '0' + i |> Char
                if (i > 9)
                    hex_dict[i] = 'A' + (i - 10) |> Char 
                end
            end
            hex_dict
        end
        
        my_code_point = let
            q = test_char_index
            remainder_array = Array{Int64,1}()
            while (q != 0)
                r = rem(q,16)
                q = div(q,16)
                push!(remainder_array,r)
            end

            my_code_point = ""
            for i ∈ reverse(remainder_array)
                tmp = hex_dict[i]
                my_code_point *= tmp |> Char
            end

            my_code_point = lpad(my_code_point, 4, '0') |> x-> "U+"*x
        end
        
        @test my_code_point == "U+1F363"
    end

    @testset "Character Set Operations" begin
        C = let
            C = Set{Char}()
            push!(C,'A')
            push!(C,'B') 
            push!(C,'Q')
            push!(C,'R')
            push!(C,'S')
            C
        end
        
        @test length(C) == 5
        @test 'A' ∈ C
        @test 'B' ∈ C
        @test 'Q' ∈ C
        @test 'Z' ∉ C  # Z should not be in the set
    end

    @testset "Emoji Variable Operations" begin
        🌽 = 16
        🍣 = 4
        
        @test 🌽 + 🍣 == 20
        @test 🌽 * 🍣 == 64
        @test 🌽 ≥ 🍣
        @test 🍣 < 🌽
    end

    @testset "String to Character Array Conversion" begin
        test_string = "Hello"
        char_array = test_string |> collect
        
        @test length(char_array) == 5
        @test char_array[1] == 'H'
        @test char_array[5] == 'o'
        @test typeof(char_array) == Array{Char,1}
    end
end;

___

## Summary
Text is numbers underneath: every character has an integer code point, and an encoding decides how that number becomes the bytes actually stored.

> __Key Takeaways:__
>
> * **A code point is not an encoding:** The integer identifying a character and the byte sequence used to store it are separate decisions, which is why the same character can occupy a different number of bytes in different encodings.
> * **Hexadecimal is for humans:** Base 16 groups four bits per digit, so it states a binary value compactly and reversibly, which is why code points, colors, and memory addresses are written that way.
> * **Collections carry the algorithm:** A set answers membership cheaply and a dictionary answers digit lookup cheaply, so choosing them well leaves the base-conversion loop short enough to read in one pass.

The `U+XXXX` string you built here is the same notation you will see in any Unicode table, and the division-and-remainder loop that produced it is the general recipe for converting between bases.
___